# Step 4 — Reusable data-quality helpers

This notebook tests the shared validation and PII-masking helpers before they are called from Airflow. The source CSV files are read only; no input file is changed.

Flow: load data → run six quality dimensions → inspect `dq_results` → inspect quarantine candidates → test masking → save local outputs.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Add the repository's include directory so the notebook uses the same helpers as Airflow.
INCLUDE_DIR = Path('include')
sys.path.insert(0, str(INCLUDE_DIR.resolve()))

from data_quality import run_quality_checks
from masking import mask_email, mask_phone

DATA_DIR = Path('include/data/nammamart')
FILES = {
    'orders': 'orders.csv',
    'orders_increment': 'orders_2026-10-01_increment.csv',
    'payments': 'payments.csv',
    'customers': 'customers.csv',
    'products': 'products.csv',
    'stores': 'stores.csv',
}

# Keeping the file-to-DataFrame mapping explicit makes the later DAG calls easy to follow.
data = {name: pd.read_csv(DATA_DIR / filename) for name, filename in FILES.items()}
for name, frame in data.items():
    print(f'{name:18} {len(frame):5,} rows loaded')

orders             2,412 rows loaded
orders_increment     160 rows loaded
payments           2,228 rows loaded
customers            405 rows loaded
products             152 rows loaded
stores                15 rows loaded


In [2]:
# Run all shared validation rules with the same identifiers that will be stored in DuckDB.
issues, dq_results = run_quality_checks(
    data,
    run_id='step4-local-test',
    dag_id='nammamart_quality_helpers_test',
)

print(f'Issue instances: {len(issues):,}')
print(f'Quality checks: {len(dq_results):,}')
print(f'Failed checks: {(dq_results["status"] == "FAIL").sum():,}')
display(dq_results[['check_name', 'dimension', 'severity', 'rows_checked', 'rows_failed', 'status']])

Issue instances: 534
Quality checks: 36
Failed checks: 31


,check_name,dimension,severity,rows_checked,rows_failed,status
0,orders.csv:customer_id:Missing customer_id,Completeness,Critical,2412,18,FAIL
1,orders.csv:order_id:Duplicate order_id,Uniqueness,Critical,2412,24,FAIL
2,orders.csv:order_status:Invalid order status,Validity,Critical,2412,3,FAIL
3,orders.csv:payment_mode:Invalid payment mode,Validity,Critical,2412,0,PASS
4,orders.csv:quantity:Quantity is not positive,Accuracy,Critical,2412,10,FAIL
5,orders.csv:order_amount:Amount does not match ...,Accuracy,Critical,2412,18,FAIL
6,orders.csv:delivery_minutes:Delivered order ha...,Completeness,Critical,2412,8,FAIL
7,orders.csv:delivery_minutes:Delivery time exce...,Accuracy,Critical,2412,6,FAIL
8,orders.csv:discount_pct:Discount outside 0-100,Validity,Critical,2412,3,FAIL
9,orders.csv:order_ts:order_ts is not a valid ti...,Validity,Critical,2412,6,FAIL


In [3]:
# Summarise failures by file, dimension and severity so the ETL DAG can route them.
display(issues.groupby(['file', 'quality_dimension', 'severity']).size().rename('issue_rows').reset_index())

# Critical rows are quarantine candidates; warnings remain loadable but are logged.
quarantine_candidates = issues[issues['severity'] == 'Critical']
warning_rows = issues[issues['severity'] == 'Warning']
print(f'Critical quarantine candidates: {len(quarantine_candidates):,}')
print(f'Warning rows to log: {len(warning_rows):,}')
display(issues.head(20))

,file,quality_dimension,severity,issue_rows
0,customers.csv,Consistency,Warning,8
1,customers.csv,Timeliness,Warning,3
2,customers.csv,Uniqueness,Critical,10
3,customers.csv,Validity,Warning,22
4,orders.csv,Accuracy,Critical,34
5,orders.csv,Completeness,Critical,26
6,orders.csv,Consistency,Critical,32
7,orders.csv,Timeliness,Critical,7
8,orders.csv,Uniqueness,Critical,24
9,orders.csv,Validity,Critical,12


Critical quarantine candidates: 200
Warning rows to log: 334


,file,row_key,column,problem,quality_dimension,proposed_rule,severity
0,orders.csv,ORD002255,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
1,orders.csv,ORD000701,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
2,orders.csv,ORD000997,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
3,orders.csv,ORD000580,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
4,orders.csv,ORD002019,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
5,orders.csv,ORD002262,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
6,orders.csv,ORD002149,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
7,orders.csv,ORD002367,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
8,orders.csv,ORD001332,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical
9,orders.csv,ORD001918,customer_id,Missing customer_id,Completeness,customer_id must be present,Critical


In [4]:
# Verify that personal data is masked before it can enter reporting tables.
mask_examples = pd.DataFrame({
    'original_email': ['priya.hegde199@example.com', 'invalid-email'],
    'masked_email': [mask_email('priya.hegde199@example.com'), mask_email('invalid-email')],
    'original_phone': ['9774349021', '123'],
    'masked_phone': [mask_phone('9774349021'), mask_phone('123')],
})
display(mask_examples)

# These files mirror the future dq_results and quarantine inputs to the pipeline.
Path('assignment').mkdir(exist_ok=True)
issues.to_csv('assignment/data_issues_log_step4.csv', index=False)
dq_results.to_csv('assignment/dq_results_step4.csv', index=False)
print('Wrote assignment/data_issues_log_step4.csv')
print('Wrote assignment/dq_results_step4.csv')

,original_email,masked_email,original_phone,masked_phone
0,priya.hegde199@example.com,pr***@example.com,9774349021,97******21
1,invalid-email,***,123,***


Wrote assignment/data_issues_log_step4.csv
Wrote assignment/dq_results_step4.csv
